In [ ]:
# generate UpSet plot, compile the dysregulated genes, proteins, phosphoproteins targeted by IPA enriched regulators

In [1]:
%load_ext rpy2.ipython

In [2]:
%%R
library(ggplot2)
library(ComplexUpset)

R[write to console]: Want to understand how all the pieces fit together? Read R for Data
Science: https://r4ds.had.co.nz/



In [3]:
import pandas as pd

In [4]:
file_d={}
file_d['AREG'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/AREG_REGULATOR_PATHWAYS.xlsx')
file_d['ATM'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/ATM_REGULATOR_PATHWAYS.xlsx')
file_d['CUL4B'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/CUL4B_REGULATOR_PATHWAYS.xlsx')
file_d['MYC'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/MYC_REGULATOR_PATHWAYS.xlsx')
file_d['Pkg'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/Pkg_REGULATOR_PATHWAYS.xlsx')
file_d['Rac'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/Rac_REGULATOR_PATHWAYS.xlsx')
file_d['ROCK'] = pd.ExcelFile('../results/IPA_downstream_analysis/regulator_pathways/ROCK_REGULATOR_PATHWAYS.xlsx')

d={k:set() for k in file_d.keys()}
for k in d.keys():
    #print(k)
    for sheet in file_d[k].sheet_names:
        df = pd.read_excel(file_d[k], sheet_name=sheet, index_col=0)
        #print(len(set(df['GENE'])))
        #print(len(d[k].union(set(df['GENE']))))
        d[k] = d[k].union(set(df['GENE']))

In [5]:
master = pd.DataFrame(columns=['Dataset']+list(d.keys()), index=[l for k in set([i for j in d.values() for i in j]) for l in [k]*4])
master['Dataset']=['6HR_MRSA', '24HR_MRSA', 'PROT', 'PHOS']*len(set([i for j in d.values() for i in j]))

In [ ]:
data_d = {'6HR_MRSA':{k:[] for k in file_d.keys()}, '24HR_MRSA':{k:[] for k in file_d.keys()}, \
          'PROT':{k:[] for k in file_d.keys()}, 'PHOS':{k:[] for k in file_d.keys()}}
for k in file_d.keys():
    for sheet in file_d[k].sheet_names:
        df = pd.read_excel(file_d[k], sheet_name=sheet, index_col=0)
        data_d['6HR_MRSA'][k]+=list(df.loc[df['DEG_6HR_MRSA_ADJPVAL']<=0.05].loc[abs(df['DEG_6HR_MRSA_LOG2FC'])>=0.585]['GENE'])
        data_d['24HR_MRSA'][k]+=list(df.loc[df['DEG_24HR_MRSA_ADJPVAL']<=0.05].loc[abs(df['DEG_24HR_MRSA_LOG2FC'])>=0.585]['GENE'])
        data_d['PROT'][k]+=list(df.loc[df['PROT_ADJPVAL']<=0.05]['GENE'])
        # data_d['PHOS'][k]+=list(df.loc[df['PHOS_ADJPVAL']<=0.05]['GENE'])

In [7]:
AREG=[]
ATM=[]
CUL4B=[]
MYC=[]
Pkg=[]
Rac=[]
ROCK=[]
for i,j in zip(master.index,master['Dataset']):
    for reg in ['AREG','ATM','CUL4B','MYC','Pkg','Rac','ROCK']:
        if reg=='AREG':
            AREG.append(i in data_d[j][reg])
        elif reg=='ATM':
            ATM.append(i in data_d[j][reg])
        elif reg=='CUL4B':
            CUL4B.append(i in data_d[j][reg])
        elif reg=='MYC':
            MYC.append(i in data_d[j][reg])
        elif reg=='Pkg':
            Pkg.append(i in data_d[j][reg])
        elif reg=='Rac':
            Rac.append(i in data_d[j][reg])
        elif reg=='ROCK':
            ROCK.append(i in data_d[j][reg])
master['AREG']=AREG
master['ATM']=ATM
master['CUL4B']=CUL4B
master['MYC']=MYC
master['Pkg']=Pkg
master['Rac']=Rac
master['ROCK']=ROCK

In [8]:
regs=['AREG','ATM','CUL4B','MYC','Pkg','Rac','ROCK']
master=master.reset_index()
drop_indx = list(master.loc[master['ATM']==False].loc[master['AREG']==False].loc[master['CUL4B']==False].loc[master['MYC']==False].loc[master['Pkg']==False].loc[master['Rac']==False].loc[master['ROCK']==False].index)
master.drop(master.index[drop_indx], inplace=True)

In [15]:
temp = master.loc[master['AREG']==False].loc\
[master['ATM']==False].loc[master['CUL4B']==False].loc\
[master['MYC']==False].loc[master['Pkg']==False].loc\
[master['Rac']==False].loc[master['ROCK']==True]
#[print(i, j) for i, j in zip(temp['index'], temp['Dataset'])]

In [18]:
%R -i master -i regs

In [19]:
%%R -w 800 -h 500

upset(
    master,
    regs,
    name='Regulator',
    annotations = list(
        'Dataset'=(
            ggplot(mapping=aes(fill=Dataset))
            + geom_bar(stat='count', position='fill')
            + scale_y_continuous(labels=scales::percent_format())
        )
    ),
    width_ratio=0.1
)
ggsave('../results/IPA_downstream_analysis/upset_plot.pdf', bg="transparent") # FIG 1

R[write to console]: Saving 11.1 x 6.94 in image

